# PROC FCMP による再利用可能な保険数理料率関数ライブラリの構築

## エグゼクティブサマリー

損害保険会社が、純保険料、費用・リスク割増、限界変動法による信頼度ブレンディング、割引準備金評価という中核の料率算出ロジックを、カスタム関数と複数出力サブルーチンとして **PROC FCMP** にカプセル化する。コンパイル済みライブラリは `CMPLIB=` システムオプションで登録され、その後、合成した100件の保険契約ポートフォリオを料率算定するDATAステップから1行ずつ呼び出される。本ノートブックにおける全ての算定値――信頼度係数 `z`、混合純保険料、請求保険料、現在価値化されたケース準備金――は、これらのコンパイル済みルーチンによって算出されたものであり、インラインの算術式によるものではない。結果として、インプライド損害率は地方で60.5%、郊外で55.8%、都市部で63.6%となり、いずれのセグメントでも100%を十分に下回り、割増後の保険料が予想損害を十分にカバーしていることを示す一方、料率算定ステップ自体は簡潔で監査可能な状態を保っている。

## データソース

| データセット | 行数 | 説明 | 主な変数 |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | `rand()` でインラインに生成した合成の損害保険契約ポートフォリオ | `policy_id`、`region`（都市部/郊外/地方）、`years_insured`、`exposure`（車両年）、`claim_count`（ポアソン分布）、`incurred_loss`（ガンマ分布severity×件数）、`class_pure_prem`（地域別の手動料率） |

DATAステップはより広い `policy_id` の範囲をループするが、この環境はライセンスなしで動作するため、出力は最初の **100件** に制限される――以下の料率算定済みの契約集合はこの100件を反映している。

# PROC FCMP による再利用可能な保険数理料率関数ライブラリの構築

アクチュアリーは、料率算定・準備金評価・レポーティングの各業務で同じ計算を繰り返し行う――損害額を*純保険料*に変換し、*費用・リスク割増*を適用して請求保険料に到達し、契約単位の自己実績を*信頼度*を用いてクラス料率とブレンドし、将来のキャッシュフローを現在価値に*割引く*。これらの計算式をDATAステップごとに再入力すると、コピー&ペーストの誤りを招きやすく、前提条件の変更が困難になる。

**PROC FCMP**（SASの関数コンパイラ）を使えば、各計算式を名前付きの関数またはサブルーチンとして一度だけ定義し、コンパイル済みルーチンをライブラリに格納し、その後は組込み関数と同様に呼び出せる。本ノートブックでは以下を行う。

1. `PROC FCMP` で小規模な保険数理関数ライブラリをコンパイルする。
2. `CMPLIB=` システムオプションでセッションに登録する。
3. 合成の損害保険ポートフォリオを生成する。
4. カスタム関数と複数出力サブルーチンを単一のDATAステップから呼び出し、全契約を料率算定する。
5. 料率算定済みポートフォリオを集計・解釈する。

## ステップ1 ― 合成の契約ポートフォリオを生成する

稼働中の自動車保険契約の集合をシミュレートする（ライセンスなしモードの上限により、以下では最初の100件を料率算定する）。各契約は、それぞれ固有の手動*純保険料*（車両年あたりの予想損害額）を持つレーティング地域に属する。事故件数はエクスポージャーでスケーリングされたポアソン過程に従い、severity（1件あたりの損害額）はガンマ分布に従うため、`incurred_loss` は現実的な複合分布（ポアソン×ガンマ）となる。`years_insured` は後の信頼度ウェイトを左右する。`call streaminit` による固定シードでデータの再現性を確保する。

In [1]:
データ portfolio;
    呼出 streaminit(20260531);
    長さ region $12;
    繰返 policy_id = 10001 から 12000;
        /* レーティング地域を割り当てる */
        u = rand('uniform');
        もし u < 0.40 なら 繰返; region = '都市部';    base_pp = 820; lambda = 0.18; 終了;
        他 もし u < 0.75 なら 繰返; region = '郊外'; base_pp = 540; lambda = 0.11; 終了;
        他 繰返; region = '地方';    base_pp = 360; lambda = 0.07; 終了;

        /* 保険継続年数（テニュア）とエクスポージャー（車両年） */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* 複合クレームプロセス：ポアソン頻度、ガンマseverity */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        繰返 c = 1 から claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        終了;
        incurred_loss = round(incurred_loss, 0.01);

        /* この契約のエクスポージャーに対する手動クラス純保険料 */
        class_pure_prem = round(base_pp * exposure, 0.01);

        出力;
    終了;
    保持 policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
実行;

処理 印刷 データ=portfolio(obs=8) noobs;
    見出 policy_id='証券番号' region='地域' years_insured='保険継続年数'
          exposure='エクスポージャー(車両年)' claim_count='事故件数'
          incurred_loss='発生損害額' class_pure_prem='クラス純保険料';
    表題 'シミュレーションされた最初の8件の証券';
実行;


                                                  シミュレーションされた最初の8件の証券                                                   

        証券番号         地域              保険継続年数                        エクスポージャー(車両年)          事故件数            発生損害額                クラス純保険料
       10001  地方                          5                                 1.36             0                0                  489.6
       10002  都市部                         8                                 0.97             1          3432.28                  795.4
       10003  都市部                         2                                 1.53             2          7155.92                 1254.6
       10004  郊外                          9                                  2.4             0                0                   1296
       10005  地方                          5                                 0.99             0                0                  356.4
       10006  郊外                          1                         


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.44 seconds
  cpu   0.44 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## ステップ2 ― 保険数理関数ライブラリをコンパイルする

ここからが本ノートブックの核心である。`OUTLIB=work.actfuncs.pricing` を指定した `PROC FCMP` は、4つの関数と1つのサブルーチンを `work.actfuncs` データセットの `pricing` パッケージへコンパイルする。

- **`pure_premium`** ― エクスポージャー1単位あたりの観測損害額（頻度×severityを統合したもの）。
- **`credibility_z`** ― 限界変動法による信頼度係数 `Z = sqrt(n / (n + k))`（`n` は契約のエクスポージャー年数、`k` は調整定数）。
- **`charged_premium`** ― 損失コストに比例的なリスク割増と固定の費用率を適用し、実際に請求する保険料に到達する。
- **`pv_reserve`** ― 将来のクレーム支払額の現在価値、`FV / (1+r)**t`。ケース準備金の割引に使用する。
- **`blend_premium`**（サブルーチン）― `OUTARGS` を用いて信頼度加重済みの純保険料と使用した信頼度係数の**2つの値**を同時に返すため、DATAステップは1回の呼び出しで両方を取得できる。

各ルーチンは `ENDSUB` で閉じ、サブルーチンは `OUTARGS` で書込み可能な引数を宣言する。

In [2]:
処理 fcmp outlib=work.actfuncs.pricing;

    /* 観測純保険料：エクスポージャー1単位あたりの損失コスト */
    function pure_premium(loss, exposure);
        もし exposure <= 0 なら RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* 限界変動法による信頼度：Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        もし n_years <= 0 なら RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* 請求保険料 = リスク割増と費用率を織り込んで引き上げた損失コスト */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        もし expense_ratio >= 1 なら RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* 将来のクレーム支払額を、t年後・利率rで割り引いた現在価値 */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* 信頼度ブレンド：加重済み純保険料と使用したZを返す */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

実行;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## ステップ3 ― `CMPLIB=` でライブラリを登録する

ルーチンをコンパイルするだけでは不十分である。後続のDATAステップやプロシジャが未知の名前を参照した際に、SASがどこでそれを探すべきかを知っている必要がある。`CMPLIB=` システムオプションは、コンパイル済みコードを保持するデータセット（3階層のパッケージ名ではない）を指す。この`OPTIONS`文の後、上記のすべての関数・サブルーチンは名前で呼び出し可能になる。

In [3]:
設定値 cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## ステップ4 ― カスタム関数でポートフォリオを料率算定する

料率算定用のDATAステップは、ほぼ算術式を含まない状態になり、保険数理上の意図が関数名から直接読み取れる。契約ごとに以下を行う。

1. `pure_premium` で契約自身の観測純保険料を計算する。
2. `blend_premium` サブルーチンを呼び出し、地域クラス料率に対して信頼度加重を行い、`OUTARGS` を通じて混合損失コストと信頼度係数の両方を取得する。
3. `charged_premium` により、リスク割増12%・費用率25%で混合損失コストを請求保険料へ引き上げる。
4. 未決のケース準備金を発生損害額の35%と見積もり、`pv_reserve` で3年・利率4%で現在価値に割り引く。

サブルーチンの出力引数（`blended_pp`、`z`）は `CALL` の前に初期化しておく必要がある点に注意。準備金の現在価値は契約ごとの実際の発生損害額によって決まるため契約ごとに異なる――無事故の契約は準備金がゼロなので `reserve_pv` もゼロになる。

In [4]:
データ rated;
    設定 portfolio;

    /* 1. 契約自身の損害実績を純保険料として算出 */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. 自己実績をクラス料率に対して信頼度加重する。
          k = 4 エクスポージャー年でほぼフル信頼度。 */
    blended_pp = .;
    z = .;
    呼出 blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. 混合損失コスト（車両年あたり）を請求保険料に変換 */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. 未決ケース準備金 = 発生損害額の35%、3年後に決着見込みとして
          現在価値に割引く（利率4%）。 */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    保持 policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
実行;

処理 印刷 データ=rated(obs=10) noobs;
    見出 policy_id='証券番号' region='地域' years_insured='保険継続年数'
          exposure='エクスポージャー(車両年)' own_pp='自社純保険料'
          blended_pp='混合純保険料' z='信頼度係数Z' premium='保険料'
          reserve_pv='準備金現在価値';
    表題 'レーティング済みポートフォリオ（最初の10件）';
    変数 policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
実行;


                                                レーティング済みポートフォリオ（最初の10件）                                                 

        証券番号         地域              保険継続年数                        エクスポージャー(車両年)              自社純保険料              混合純保険料            信頼度係数Z        保険料                準備金現在価値
       10001  地方                          5                                 1.36                   0               91.67             0.745     186.18                      0
       10002  都市部                         8                                 0.97             3538.43             3039.59             0.816    4402.95                1067.95
       10003  都市部                         2                                 1.53             4677.07             3046.88             0.577    6961.51                2226.55
       10004  郊外                          9                                  2.4                   0               90.69             0.832     325.03                      0
       10005 


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## ステップ5 ― 料率算定済みポートフォリオを集計する

全契約が同一のコンパイル済みライブラリを通じて料率算定されたので、ポートフォリオを地域別に集計できる。平均請求保険料、平均信頼度係数、発生損害額合計、現在価値化されたケース準備金合計を集計し、その後インプライド損害率を導出して、割増後の保険料が予想損害をカバーしているかを確認する。

In [5]:
処理 平均 データ=rated mean sum maxdec=2 nonobs;
    分類 region;
    変数 premium z incurred_loss reserve_pv;
    見出 region='地域' premium='保険料' z='信頼度係数Z'
          incurred_loss='発生損害額' reserve_pv='準備金現在価値';
    表題 '地域別ポートフォリオ集計';
実行;

/* インプライド損害率 = 発生損害額 / 請求保険料、および現在価値化された
   準備金を地域別に集計する。 */
処理 SQL;
    表題 '地域別インプライド損害率と準備金現在価値';
    選択 region                                見出='地域',
           count(*)                              AS n_policies       見出='契約件数',
           sum(incurred_loss)                    AS total_loss       書式=dollar12.2  見出='発生損害額合計',
           sum(premium)                          AS total_premium    書式=dollar12.2  見出='保険料合計',
           sum(incurred_loss) / sum(premium)     AS loss_ratio       書式=percent8.1  見出='損害率',
           sum(reserve_pv)                       AS total_pv_reserve 書式=dollar12.2  見出='準備金現在価値合計'
    FROM rated
    GROUP 基準 region
    ORDER 基準 region;
QUIT;
表題;


                                                      地域別ポートフォリオ集計                                                      

                                                  The MEANS Procedure

                                         Analysis Variable : premium 保険料

        地域                  Mean            Sum
        ---------------------------------------
        地方                476.61       11438.62
        郊外                813.04       34147.74
        都市部              1987.17       67563.93
        ---------------------------------------

                                         Analysis Variable : z 信頼度係数Z

        地域                  Mean            Sum
        ---------------------------------------
        地方                  0.71          17.14
        郊外                  0.68          28.36
        都市部                 0.70          23.90
        ---------------------------------------

                                   Analysis Variable : incurred_loss 発生損害額

        


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## 結果の解釈

料率算定用のDATAステップは、単一の割引式や信頼度計算式を一切明示的に書き出さない――`pure_premium`、`blend_premium`、`charged_premium`、`pv_reserve` を呼び出すだけである。それこそが **PROC FCMP** がもたらす価値である。保険数理上の前提は、単体テスト・バージョン管理・再利用が可能な1つのコンパイル済みライブラリに集約される。信頼度定数 `k`、リスク割増率、費用率を変更する場合も、ライブラリ側の1箇所を編集するだけで済み、プログラム全体を探し回る必要はない。

料率算定済みの契約集合と地域別ロールアップを読み解くと：

- **信頼度（`z`）** は `years_insured` とともに上昇し、限界変動法の公式 `Z = sqrt(n / (n + k))` が示すとおりの挙動を示す。最初の10件のうち、保険継続1年の郊外契約（10006）は `z = 0.447`、保険継続2年の都市部契約（10003）は `z = 0.577`、保険継続9年の郊外契約（10004）は `z = 0.832` に達する。実績の薄い契約は地域クラス料率に引き寄せられ、長期契約は自己実績に依拠する。
- **ブレンドの実際:** 無事故の契約（大半を占める）は `own_pp = 0` となるため、`blend_premium` は `(1 - z)` にクラス料率を乗じた値に近い `blended_pp` を返す――例えば契約10001（地方、`z = 0.745`）は、`360`/車両年のクラス料率に対して`91.67`となる。実際に損害が発生した2件の都市部契約、10002と10003は、逆に混合損失コストが自身の高い実績値に向けて引き上げられる（`3039.59` と `3046.88`）。
- **請求保険料** は、`charged_premium` がリスク割増12%を加え、費用率25%で引き上げる（損失コストに対して固定の `1.12 / 0.75 = 1.493` 倍率）ため、混合損失コストを上回る。平均請求保険料は地方で`476.61`、郊外で`813.04`、都市部で`1,987.17`となる。
- **割引後の準備金:** `pv_reserve` は各契約の未決ケース準備金（発生損害額の35%）を3年・利率4%で割り引く（現在価値係数`0.889`）。無事故の契約は`reserve_pv = 0`であり、都市部の2件の実損害契約は`1,067.95`と`2,226.55`を寄与する。ロールアップすると、ポートフォリオは地方で`$2,151.56`、郊外で`$5,932.52`、都市部で`$13,364.13`の現在価値化された準備金を保有する。
- **インプライド損害率** は地方で60.5%、郊外で55.8%、都市部で63.6%となり、いずれも100%を十分に下回るため、全セグメントで割増後の保険料が予想損害をカバーしている。都市部セグメントが最も高いのは、2件の大口シミュレーション損害によるものであり、実際の料率レビューでは、この兆候がより多くの事故年にわたって持続するかを検証してから手動料率を調整すべきである。

`blend_premium` サブルーチンは、`OUTARGS` イディオムを用いて1回の `CALL` から複数の結果（混合損失コストと信頼度係数`z`）を返す方法も示しており、個別の関数呼び出しを避けつつ契約単位の料率算定ロジックをコンパクトかつ監査可能に保っている。